# chesszero — B1 : pré-entraînement du réseau 12 × 160 sur 48 M de positions (Kaggle)

Plan approuvé le 2026-11-08 (`docs/plan_b1.md`, journal « Plan B1 approuvé ») : réseau 12 blocs × 160 canaux entraîné
de zéro avec la recette de `kaggle_sup` (92 000 pas de 1 024) sur le Dataset `lichess-48m` (16 mois Lichess 2000+,
positions des ouvertures protégées retirées de l'entraînement).

**Trois modes**, choisis dans la cellule 0 (`MODE`) :
- `"check"` (**défaut, sans GPU, rien n'est écrit dans `/kaggle/working`**) : vérifie le code (`COMMIT`), le Dataset (chaque
  shard : SHA-256, taille, positions), la sortie précédente éventuelle (exactement zéro ou une, au pas `EXPECTED_STEP`) et
  dit ce que la session ferait. Quelques minutes, sans quota GPU.
- `"bench"` : essai de débit seul (~6 min) : 300 pas à 1 GPU puis à 2 GPU (DataParallel), contrôle de cohérence des deux,
  choix du nombre de GPU ; rien n'est entraîné pour de bon.
- `"train"` : l'entraînement (reprend tout seul depuis la sortie précédente). Avec `GPUS = "auto"`, l'essai de débit est
  fait d'abord et choisit 1 ou 2 GPU (2 seulement s'ils sont cohérents et au moins 10 % plus rapides).
  **Garde-fou** (critère approuvé) : si le débit est sous 3 000 positions/s au pas 2 000, arrêt avec `last.pt`
  sauvegardé ; la version reste « Successful » (bandeau final) ; ne pas reprendre sans décision.

**Réglages de la session** : *Accelerator* = **GPU T4 x2** (`train`, `bench`) ou **None** (`check`) ; *Internet* = **On** ;
*Persistence* = No persistence.

**Entrées** (*Add Input*) : le Dataset privé **`lichess-48m`** ; à partir de la 2e session, **la sortie de la session
précédente de ce notebook** (elle contient `b1_12x160/last.pt`), une seule. Rien d'autre n'est nécessaire.

**`EXPECTED_STEP`** = pas de la sortie précédente (0 pour la première session) ; la dernière cellule donne la valeur
suivante.

**Sortie** (`/kaggle/working/b1_12x160`) : `last.pt`, `best.pt`, 3 derniers `ckpt_*.pt`, `metrics.csv`, `bench.json`,
`throughput_check.json`. Le dépôt est cloné dans `/tmp` (hors de la sortie).

In [ ]:
# 0. Réglages ---------------------------------------------------------------------------------------------
MODE = "check"            # "check" (défaut, sans GPU, rien d'écrit) | "bench" | "train"
COMMIT = "0000000000000000000000000000000000000000"     # hash COMPLET du code utilisé : à changer volontairement, jamais None
EXPECTED_STEP = 0         # pas de la sortie précédente en entrée (0 = première session, départ de zéro)
GPUS = "auto"             # train : "auto" (essai de débit puis choix), 1 ou 2
FULL_MINUTES = 630        # train : temps d'entraînement de la session (hors chargement ~10 min, essai ~6 min, vérifications)
ACCEPT_LOW_THROUGHPUT = False   # True seulement sur décision, pour reprendre un entraînement arrêté par le garde-fou de débit
CONFIG = "kaggle_l"       # configs/kaggle_l.toml : 12 blocs x 160, garde-fou 3 000 positions/s au pas 2 000

REPO_URL = "https://github.com/Enz091/Project_chess.git"

import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REPO_DIR = Path("/tmp/Project_chess")   # hors de /kaggle/working : pas dans la sortie
OUT = WORKING / "b1_12x160"             # NE PAS renommer : la reprise cherche « b1_12x160/last.pt »
assert MODE in ("check", "bench", "train"), MODE
assert GPUS in ("auto", 1, 2), GPUS
assert 30 <= FULL_MINUTES <= 640, "FULL_MINUTES entre 30 et 640 (coupure de Kaggle à 12 h)"
if not re.fullmatch(r"[0-9a-f]{40}", str(COMMIT)):
    raise ValueError("COMMIT doit être un hash complet de 40 caractères hexadécimaux (pas None, pas « main »)")


def run(cmd, cwd=None, ok=(0,)):
    # Lance une commande et affiche sa sortie au fil de l'eau ; lève une erreur si son code n'est pas dans `ok`.
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if code not in ok:
        raise RuntimeError(f"échec ({code}) : {' '.join(str(c) for c in cmd)}")
    return code


print(f"MODE = {MODE} ; COMMIT = {COMMIT} ; EXPECTED_STEP = {EXPECTED_STEP} ; GPUS = {GPUS}")

In [ ]:
# 1. Dépôt au commit figé et installation (sans réinstaller torch) -------------------------------------------------
import importlib
import importlib.metadata as md

torch_before = md.version("torch")
if (REPO_DIR / ".git").exists():
    run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])
run(["git", "checkout", "--quiet", "--detach", COMMIT], cwd=REPO_DIR)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"le dépôt est sur {head}, pas sur COMMIT {COMMIT}"
run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])  # pas de torch dans les dépendances
sys.path.insert(0, str(REPO_DIR / "src"))  # le noyau ne voit pas un paquet installé après son démarrage
importlib.invalidate_caches()
run(["git", "log", "-1", "--format=commit %H%n%s"], cwd=REPO_DIR)
assert md.version("torch") == torch_before, "pip a modifié torch : à ne pas faire sur Kaggle"
print("torch inchangé :", torch_before)

In [ ]:
# 2. Environnement ------------------------------------------------------------------------------------------------
import platform

import torch

print("Python :", platform.python_version(), "| torch :", torch.__version__, "| CUDA :", torch.version.cuda)
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
WORKERS = max(1, min(3, cores - 1))   # processus des masques de coups légaux ; un cœur pour l'entraînement
print(f"Cœurs CPU : {cores} -> {WORKERS} processus de masques")
if Path("/proc/meminfo").exists():
    mem = {l.split(":")[0]: int(l.split()[1]) for l in Path("/proc/meminfo").read_text().splitlines()[:3]}
    print(f"RAM : {mem['MemTotal'] / 1e6:.0f} Go (libre {mem['MemAvailable'] / 1e6:.0f} Go) ; "
          f"disque /kaggle/working : {shutil.disk_usage(WORKING).free / 1e9:.0f} Go libres")
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
if MODE == "check":
    print(f"Mode check : GPU non requis ({n_gpu} visible(s))")
elif n_gpu == 0:
    raise RuntimeError("Aucun GPU : Session options -> Accelerator -> GPU T4 x2")
else:
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i} : {p.name}, {p.total_memory / 1e9:.0f} Go")
    if GPUS == 2 and n_gpu < 2:
        raise RuntimeError("GPUS = 2 mais un seul GPU visible : choisir GPU T4 x2 ou GPUS = \"auto\"")

In [ ]:
# 3. Entrées : Dataset lichess-48m (chaque shard vérifié) et sortie précédente --------------------------------------
from chesszero.pretrain_inputs import check_pretrain_inputs

def progress(i, n):
    if i % 25 == 0 or i == n:
        print(f"  vérification des shards : {i}/{n}", flush=True)

INPUTS = check_pretrain_inputs(INPUT_ROOT, CONFIG, EXPECTED_STEP, ACCEPT_LOW_THROUGHPUT, progress=progress)
for note in INPUTS["notes"]:
    print(note)
for err in INPUTS["errors"]:
    print("REFUS :", err)
DATA = INPUTS["data"]
done = INPUTS["previous_step"] or 0
left = max(0, INPUTS["total_steps"] - done)
print(f"Pas restants : {left:,} sur {INPUTS['total_steps']:,} ; à 4 300 positions/s (1 T4, estimation) ≈ "
      f"{left * 1024 / 4300 / 3600:.1f} h ; à 7 000 (2 T4) ≈ {left * 1024 / 7000 / 3600:.1f} h ; "
      f"cette session : au plus {FULL_MINUTES} min d'entraînement")
if not INPUTS["ok"]:
    raise RuntimeError("VÉRIFICATION DES ENTRÉES : la session NE démarrerait PAS (voir REFUS ci-dessus)")
print("VÉRIFICATION DES ENTRÉES : la session démarrerait.")
if MODE == "check":
    print(f"/kaggle/working après la vérification : {sorted(p.name for p in WORKING.iterdir()) or 'vide'}")

In [ ]:
# 4. Mode bench : essai de débit 1 GPU / 2 GPU (rien n'est entraîné pour de bon) -------------------------------------
TRAIN = REPO_DIR / "scripts" / "train.py"
if MODE == "bench":
    run([sys.executable, TRAIN, "--config", CONFIG, "--data", DATA, "--out", WORKING / "b1_bench",
         "--workers", WORKERS, "--bench"])
else:
    print("MODE =", MODE, ": pas d'essai de débit seul")

In [ ]:
# 5. Mode train : entraînement (reprise automatique depuis la sortie précédente) ---------------------------------------
import json

from chesszero.pretrain_inputs import restore_previous

END = None
if MODE == "train":
    if INPUTS["previous"] is not None:
        print("Reprise : fichiers copiés depuis la sortie précédente :", restore_previous(INPUTS["previous"], OUT))
    if done >= INPUTS["total_steps"]:
        END = "déjà terminé"
        print("Entraînement déjà terminé : rien à faire.")
    else:
        code = run([sys.executable, TRAIN, "--config", CONFIG, "--data", DATA, "--out", OUT, "--workers", WORKERS,
                    "--max-minutes", FULL_MINUTES, "--gpus", GPUS], ok=(0, 6))
        END = "garde-fou de débit" if code == 6 else "fin normale (budget ou terminé)"
        run([sys.executable, REPO_DIR / "scripts" / "summarize_run.py", OUT, "--skip", 5], ok=(0, 1, 2))
else:
    print("MODE =", MODE, ": pas d'entraînement")

In [ ]:
# 6. Bilan de la session et réglage de la suivante ------------------------------------------------------------------
if MODE == "train" and (OUT / "last.pt").exists():
    step = torch.load(OUT / "last.pt", map_location="cpu", weights_only=False)["step"]
    if (OUT / "bench.json").exists():
        b = json.loads((OUT / "bench.json").read_text(encoding="utf-8"))
        print(f"Essai de débit : {b['why']}")
    if (OUT / "throughput_check.json").exists():
        c = json.loads((OUT / "throughput_check.json").read_text(encoding="utf-8"))
        print(f"Contrôle du débit au pas {c['step']} : {c['pos_per_s']:,.0f} positions/s (seuil {c['min_pos_per_s']:,.0f}) -> "
              f"{'OK' if c['ok'] else 'ÉCHEC'}")
    print(f"Fin : {END} ; pas atteint {step:,} / {INPUTS['total_steps']:,}")
    if END == "garde-fou de débit":
        print("=" * 100 + "\nARRÊT PAR LE GARDE-FOU DE DÉBIT (critère approuvé) : ce n'est pas une erreur. Ne pas relancer "
              "sans décision.\n" + "=" * 100)
    elif step < INPUTS["total_steps"]:
        print(f"SESSION SUIVANTE : ajouter la sortie de CETTE version en entrée (retirer l'ancienne) et mettre "
              f"EXPECTED_STEP = {step}")
    else:
        print("ENTRAÎNEMENT TERMINÉ : télécharger b1_12x160/best.pt et metrics.csv (critères 2 à 4 sur le PC).")
    print("Contenu de la sortie :", sorted(p.name for p in OUT.iterdir()))